In [7]:
%pip install -q -U groq pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


In [8]:
from groq import Groq
from getpass import getpass
GROQ_API_KEY = getpass("Enter your Groq API Key: ")

client = Groq(api_key=GROQ_API_KEY)

available_models = {
    model.id for model in client.models.list().data
}

preferred_models = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b"
]

MODEL_NAME = next(
    (model for model in preferred_models
     if model in available_models),
    None
)

if MODEL_NAME is None:
    print("No preferred model is available.")
    print("Available models:")
    print(sorted(available_models))
    raise RuntimeError(
        "Please select a supported chat model from the list."
    )

print("Groq connected successfully!")
print("Selected model:", MODEL_NAME)

test_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Reply with only: Connection successful"
        }
    ],
    temperature=0,
    max_completion_tokens=100
)

print("\nModel test:")
print(test_response.choices[0].message.content)

Enter your Groq API Key: ··········
Groq connected successfully!
Selected model: openai/gpt-oss-120b

Model test:
Connection successful


In [9]:
import sqlite3
import os
import json

DB_NAME = "business.db"

if os.path.exists(DB_NAME):
    os.remove(DB_NAME)

conn = sqlite3.connect(DB_NAME)
conn.execute("PRAGMA foreign_keys = ON")

cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    salary REAL,
    department_id INTEGER,
    FOREIGN KEY (department_id)
        REFERENCES departments(department_id)
);

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    amount REAL,
    order_date TEXT,
    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
);
""")

cursor.executemany(
    "INSERT INTO departments VALUES (?, ?)",
    [
        (1, "IT"),
        (2, "HR"),
        (3, "Finance")
    ]
)

cursor.executemany(
    "INSERT INTO employees VALUES (?, ?, ?, ?)",
    [
        (1, "Rahul", 60000, 1),
        (2, "Priya", 75000, 1),
        (3, "Arjun", 45000, 2),
        (4, "Sneha", 80000, 3)
    ]
)

cursor.executemany(
    "INSERT INTO customers VALUES (?, ?, ?)",
    [
        (1, "Customer A", "Hyderabad"),
        (2, "Customer B", "Bangalore"),
        (3, "Customer C", "Chennai")
    ]
)

cursor.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?)",
    [
        (101, 1, 5000, "2026-01-10"),
        (102, 2, 8000, "2026-01-12"),
        (103, 1, 3000, "2026-01-15"),
        (104, 3, 6000, "2026-01-20")
    ]
)

conn.commit()

print("Business database created successfully!")

tables = conn.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
AND name NOT LIKE 'sqlite_%'
ORDER BY name
""").fetchall()

print("\nAvailable tables:")
for table in tables:
    print("-", table[0])

def get_database_schema(connection):

    schema = {}

    for (table_name,) in tables:

        columns = connection.execute(
            f'PRAGMA table_info("{table_name}")'
        ).fetchall()

        foreign_keys = connection.execute(
            f'PRAGMA foreign_key_list("{table_name}")'
        ).fetchall()

        schema[table_name] = {
            "columns": [
                {
                    "name": column[1],
                    "type": column[2]
                }
                for column in columns
            ],
            "foreign_keys": [
                {
                    "from": fk[3],
                    "references_table": fk[2],
                    "references_column": fk[4]
                }
                for fk in foreign_keys
            ]
        }

    return schema


DATABASE_SCHEMA = get_database_schema(conn)

print("\nDATABASE SCHEMA:")
print(json.dumps(DATABASE_SCHEMA, indent=4))

Business database created successfully!

Available tables:
- customers
- departments
- employees
- orders

DATABASE SCHEMA:
{
    "customers": {
        "columns": [
            {
                "name": "customer_id",
                "type": "INTEGER"
            },
            {
                "name": "customer_name",
                "type": "TEXT"
            },
            {
                "name": "city",
                "type": "TEXT"
            }
        ],
        "foreign_keys": []
    },
    "departments": {
        "columns": [
            {
                "name": "department_id",
                "type": "INTEGER"
            },
            {
                "name": "department_name",
                "type": "TEXT"
            }
        ],
        "foreign_keys": []
    },
    "employees": {
        "columns": [
            {
                "name": "employee_id",
                "type": "INTEGER"
            },
            {
                "name": "employee_name",
     

In [10]:
REFINEMENT_PROMPT = """
You are an expert business analyst and question refinement assistant.

Your task is to transform a generic business question into
a detailed, context-specific database question.

AVAILABLE DATABASE:
business.db

AVAILABLE BUSINESS CONTEXT:

1. departments
   - department_id
   - department_name

2. employees
   - employee_id
   - employee_name
   - salary
   - department_id

3. customers
   - customer_id
   - customer_name
   - city

4. orders
   - order_id
   - customer_id
   - amount
   - order_date

DATABASE RELATIONSHIPS:
- employees.department_id references departments.department_id
- orders.customer_id references customers.customer_id

INSTRUCTIONS:

1. Understand the generic business question.
2. Identify the relevant business entities.
3. Identify the required metrics, filters and conditions.
4. Identify the relevant tables and columns.
5. Convert the question into a specific database question.
6. Always identify the target database as business.db.
7. Never invent tables, columns, companies or business facts.
8. Do not assume missing dates or filters.
9. If the question cannot be answered using this schema,
   clearly mention the missing information.
10. If clarification is necessary, populate clarification_needed.

Return ONLY a valid JSON object using these fields:

{
    "refined_question": "Specific business question",
    "target_database": "business.db",
    "company_scope": "Business context or relevant entity scope",
    "relevant_tables": [],
    "assumptions": [],
    "clarification_needed": [],
    "is_answerable": true
}

Set is_answerable to false if the available schema
cannot answer the question.
"""


def refine_question(generic_question):

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": REFINEMENT_PROMPT
            },
            {
                "role": "user",
                "content": (
                    "Refine this generic business question:\n"
                    + generic_question
                )
            }
        ],
        response_format={
            "type": "json_object"
        },
        temperature=0,
        max_completion_tokens=1500
    )

    response_text = response.choices[0].message.content

    refined_data = json.loads(response_text)

    required_fields = [
        "refined_question",
        "target_database",
        "company_scope",
        "relevant_tables",
        "assumptions",
        "clarification_needed",
        "is_answerable"
    ]

    for field in required_fields:
        if field not in refined_data:
            raise ValueError(
                f"Missing JSON field: {field}"
            )

    if not isinstance(refined_data["is_answerable"], bool):
        raise ValueError("is_answerable must be Boolean.")

    return refined_data


generic_question = "Show the average salary in each department."

refined_output = refine_question(generic_question)

print("STAGE 1 - GENERIC QUESTION")
print(generic_question)

print("\nSTAGE 1 - REFINED JSON")
print(json.dumps(refined_output, indent=4))

STAGE 1 - GENERIC QUESTION
Show the average salary in each department.

STAGE 1 - REFINED JSON
{
    "refined_question": "What is the average salary for each department?",
    "target_database": "business.db",
    "company_scope": "Department salary analysis across all departments",
    "relevant_tables": [
        "employees",
        "departments"
    ],
    "assumptions": [],
    "clarification_needed": [],
    "is_answerable": true
}


In [12]:
import re
import json
SQL_GENERATION_PROMPT = """
You are an expert Text-to-SQL developer.

Your task is to generate accurate SQLite queries
using the refined business question and database schema.

STRICT RULES:

1. Generate only SQLite-compatible SQL.
2. Use only tables and columns in the provided schema.
3. Follow foreign-key relationships when joining tables.
4. Use appropriate WHERE, GROUP BY, ORDER BY and JOIN clauses.
5. Use aggregate functions such as SUM, AVG, COUNT and MAX
   when required.
6. Apply date filters only when specified or established
   by the refined question.
7. Never invent database tables or columns.
8. Generate only SELECT or WITH queries.
9. Never generate INSERT, UPDATE, DELETE, DROP, ALTER or CREATE.
10. Do not generate PRAGMA or ATTACH statements.
11. Return only the SQL query.
12. Do not include explanations or Markdown code fences.

If the refined question cannot be answered using the schema,
do not generate a query. Return an error instead.

Expected response:
Only the SQL query.
"""


def generate_sql(refined_data, database_schema):

    if not refined_data["is_answerable"]:
        raise ValueError(
            "Question cannot be answered using the available schema: "
            + str(refined_data["clarification_needed"])
        )

    if refined_data["target_database"] != DB_NAME:
        raise ValueError("Unknown target database.")

    prompt = f"""
REFINED BUSINESS QUESTION:
{refined_data['refined_question']}

TARGET DATABASE:
{refined_data['target_database']}

BUSINESS SCOPE:
{refined_data['company_scope']}

RELEVANT TABLES:
{json.dumps(refined_data['relevant_tables'])}

ASSUMPTIONS:
{json.dumps(refined_data['assumptions'])}

DATABASE SCHEMA:
{json.dumps(database_schema, indent=4)}

Generate the appropriate SQLite SELECT query.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SQL_GENERATION_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=1500
    )

    sql_query = response.choices[0].message.content.strip()

    sql_query = re.sub(
        r"^```(?:sql)?\s*|\s*```$",
        "",
        sql_query,
        flags=re.IGNORECASE
    ).strip()

    if not sql_query:
        raise ValueError("The LLM returned an empty SQL query.")

    return sql_query

generated_sql = generate_sql(
    refined_output,
    DATABASE_SCHEMA
)

print("STAGE 2 - REFINED QUESTION")
print(refined_output["refined_question"])

print("\nGENERATED SQL QUERY:")
print(generated_sql)

STAGE 2 - REFINED QUESTION
What is the average salary for each department?

GENERATED SQL QUERY:
SELECT d.department_name, AVG(e.salary) AS average_salary
FROM employees e
JOIN departments d ON e.department_id = d.department_id
GROUP BY d.department_id, d.department_name
ORDER BY d.department_name;


In [13]:
import re
import time
import sqlite3
import json

def execute_safe_sql(sql_query):

    sql = sql_query.strip()

    if not re.match(r"^(SELECT|WITH)\b", sql, re.IGNORECASE):
        raise ValueError("Only SELECT or WITH queries are allowed.")

    if ";" in sql.rstrip().rstrip(";"):
        raise ValueError("Multiple SQL statements are not allowed.")

        db_path = os.path.abspath(DB_NAME)

    read_conn = sqlite3.connect(
        f"file:{db_path}?mode=ro",
        uri=True
    )

    allowed_actions = {
        sqlite3.SQLITE_SELECT,
        sqlite3.SQLITE_READ,
        sqlite3.SQLITE_FUNCTION,
        sqlite3.SQLITE_RECURSIVE
    }

    def safety_guard(action, arg1, arg2, database, trigger):

        if action in allowed_actions:
            return sqlite3.SQLITE_OK

        return sqlite3.SQLITE_DENY

    try:

        read_conn.set_authorizer(safety_guard)

        start_time = time.perf_counter()

        cursor = read_conn.execute(sql)

        rows = cursor.fetchall()

        columns = [
            item[0] for item in cursor.description
        ]

        execution_time = time.perf_counter() - start_time

        query_plan = read_conn.execute(
            "EXPLAIN QUERY PLAN " + sql
        ).fetchall()

        return {
            "columns": columns,
            "rows": rows,
            "execution_time": execution_time,
            "query_plan": query_plan
        }

    finally:
        read_conn.close()



def text_to_sql_pipeline(generic_question):

    print("\n" + "=" * 65)
    print("GENERIC QUESTION:", generic_question)
    print("=" * 65)

    try:

        refined_data = refine_question(generic_question)

        print("\nSTAGE 1 - REFINED JSON:")
        print(json.dumps(refined_data, indent=4))

        if not refined_data["is_answerable"]:

            print("\nQuestion cannot be answered.")
            print("Clarification required:")
            print(refined_data["clarification_needed"])

            return {
                "generic_question": generic_question,
                "refined_json": refined_data,
                "generated_sql": None,
                "query_results": [],
                "status": "Clarification required"
            }

        sql_query = generate_sql(
            refined_data,
            DATABASE_SCHEMA
        )

        print("\nSTAGE 2 - GENERATED SQL:")
        print(sql_query)

        result = execute_safe_sql(sql_query)

        print("\nQUERY RESULTS:")
        print(" | ".join(result["columns"]))

        for row in result["rows"]:
            print(row)

        print(
            "\nExecution Time:",
            round(result["execution_time"], 6),
            "seconds"
        )


        print("\nQUERY EXECUTION PLAN:")

        for item in result["query_plan"]:
            print(item)

        print("\nOPTIMIZATION SUGGESTIONS:")

        suggestions = []

        if re.search(r"\bSELECT\s+\*", sql_query, re.I):
            suggestions.append(
                "Avoid SELECT * and retrieve only required columns."
            )

        if re.search(r"\bJOIN\b", sql_query, re.I):
            suggestions.append(
                "Check indexes on columns used in JOIN conditions."
            )

        if any(
            "SCAN " in str(item).upper()
            for item in result["query_plan"]
        ):
            suggestions.append(
                "Review table scans and consider suitable indexes."
            )

        if not suggestions:
            suggestions.append(
                "No obvious optimization suggestions identified."
            )

        for suggestion in suggestions:
            print("-", suggestion)

        return {
            "generic_question": generic_question,
            "refined_json": refined_data,
            "generated_sql": sql_query,
            "query_results": result["rows"],
            "execution_time": result["execution_time"],
            "query_plan": result["query_plan"],
            "status": "Success"
        }

    except Exception as error:

        print("\nPIPELINE ERROR:", str(error))

        return {
            "generic_question": generic_question,
            "refined_json": None,
            "generated_sql": None,
            "query_results": [],
            "status": f"Failed: {str(error)}"
        }

test_questions = [

    "Find the employee with the highest salary.",

    "Show the average salary in each department.",

    "Find customers from Hyderabad.",

    "Show the total order amount for each customer."
]

results = []

for question in test_questions:

    output = text_to_sql_pipeline(question)

    results.append(output)


print("\n" + "=" * 65)
print("FINAL PIPELINE SUMMARY")
print("=" * 65)

for item in results:

    print("\nQuestion:", item["generic_question"])
    print("Status:", item["status"])



documentation = """
# Two-Stage Prompt Engineering for Text-to-SQL

## 1. Project Overview

This project converts generic business questions into
context-specific questions and generates SQL queries
using a Large Language Model.

The implementation contains two stages:

- Stage 1: Generic Question Refinement into JSON.
- Stage 2: SQL Query Generation using Database Schema.

Technology Stack:
- Python
- Groq API
- OpenAI GPT-OSS model hosted on Groq
- SQLite
- JSON

## 2. Database Design

Database: business.db

Tables:
- departments
- employees
- customers
- orders

The schema is retrieved dynamically from SQLite,
including column names and foreign-key relationships.

## 3. Stage 1 - Question Refinement

### Prompt Design

The first prompt provides the LLM with business context,
available tables, columns, and database relationships.

It instructs the model to identify relevant business entities,
metrics, filters, and the target database.

### Output Format

The output is a JSON object containing:

- refined_question
- target_database
- company_scope
- relevant_tables
- assumptions
- clarification_needed
- is_answerable

### Rationale

Structured JSON makes the output easy to parse using Python.
Providing the actual database schema reduces ambiguity.
The model is instructed not to invent missing information.

## 4. Stage 2 - SQL Generation

### Prompt Design

The second prompt receives the refined question,
target database, assumptions, and actual database schema.

It instructs the model to generate SQLite-compatible,
read-only SELECT queries.

### Rationale

Providing schema information helps the LLM identify
valid tables, columns, and foreign-key relationships.
SQL restrictions reduce the risk of unwanted database changes.

## 5. Prompt Iterations and Refinements

Initial approach:
The original notebook generated SQL directly from generic questions.

Identified limitation:
The system did not separately identify the business context
or produce a refined JSON question.

Refinement 1:
Introduced a dedicated question-refinement prompt.

Refinement 2:
Added structured JSON output containing the refined question,
database name, relevant tables, assumptions, and clarification status.

Refinement 3:
Provided actual database schema and foreign-key relationships
to the SQL generation prompt.

Refinement 4:
Added SQL validation, read-only execution, and query
execution-plan inspection.

## 6. Example Questions and Results

"""

for item in results:

    documentation += f"""
### Generic Question

{item['generic_question']}

### Refined JSON

{json.dumps(item['refined_json'], indent=4) if item['refined_json'] else 'Not generated'}

### Generated SQL

{item['generated_sql'] if item['generated_sql'] else 'Not generated'}

### Query Results

{json.dumps(item['query_results'], indent=4)}

### Execution Status

{item['status']}

---
"""

documentation += """
## 7. Effectiveness Analysis

### Advantages

- Converts generic questions into detailed questions.
- Identifies the target database in JSON.
- Uses actual database schema for SQL generation.
- Supports SQL joins and aggregate operations.
- Executes generated queries against SQLite.
- Displays query results and execution plans.

### Challenges

- Generic questions may lack sufficient business context.
- Missing information may require clarification.
- The LLM may generate incorrect SQL joins or filters.
- A syntactically valid query may still return incorrect results.
- Model availability and API limits can affect execution.

### Improvements

- Add more company-specific database schemas.
- Implement a clarification and retry mechanism.
- Validate generated queries against expected results.
- Add automated SQL correction.
- Use larger and more realistic business datasets.

## 8. Conclusion

The project demonstrates a two-stage Text-to-SQL workflow
using prompt engineering.

Stage 1 converts generic questions into structured JSON.
Stage 2 generates SQL queries using the refined question
and database schema.

The generated SQL is validated and executed against
the business database to retrieve relevant information.
"""

with open(
    "Text_to_SQL_Project_Documentation.md",
    "w",
    encoding="utf-8"
) as file:

    file.write(documentation)

print("\nProject documentation created successfully!")

print("File: Text_to_SQL_Project_Documentation.md")


GENERIC QUESTION: Find the employee with the highest salary.

STAGE 1 - REFINED JSON:
{
    "refined_question": "Which employee has the highest salary, and what is their salary amount?",
    "target_database": "business.db",
    "company_scope": "employees",
    "relevant_tables": [
        "employees"
    ],
    "assumptions": [],
    "clarification_needed": [],
    "is_answerable": true
}

STAGE 2 - GENERATED SQL:
SELECT employee_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 1;

QUERY RESULTS:
employee_name | salary
('Sneha', 80000.0)

Execution Time: 0.000277 seconds

QUERY EXECUTION PLAN:
(4, 0, 0, 'SCAN employees')
(17, 0, 0, 'USE TEMP B-TREE FOR ORDER BY')

OPTIMIZATION SUGGESTIONS:
- Review table scans and consider suitable indexes.

GENERIC QUESTION: Show the average salary in each department.

STAGE 1 - REFINED JSON:
{
    "refined_question": "What is the average salary for each department?",
    "target_database": "business.db",
    "company_scope": "Employees and D